In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

STORAGE_ACCOUNT = "dltlearn"
RAW_CONTAINER = "raw"
bronze_path = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bronze"
silver_path = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver"

catalog = 'test'
schema = 'silver'

database = f"{catalog}.{schema}"

def reading_bronze_files(table_name: str):
    try:
        df = spark.read.format('delta').load(f"{bronze_path}/{table_name}")
        rows = df.count()
        print(f"rows read are {rows}")
        string_cols = [c.name for c in df.schema.fields if str(c.dataType) == "StringType()"]
        for c in string_cols:
          df = df.withColumn(c, trim(col(c)))
        
        if table_name =='orders':
            df = df.withColumn("order_purchase_timestamp",      to_timestamp(col("order_purchase_timestamp"))) \
                   .withColumn("order_approved_at",             to_timestamp(col("order_approved_at"))) \
                   .withColumn("order_delivered_carrier_date",  to_timestamp(col("order_delivered_carrier_date"))) \
                   .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
                   .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
                   .withColumn("is_delivered",when(col('order_status') == 'delivered',True).otherwise(False))\
                   .withColumn("delivery_delay_days",when(col("order_delivered_customer_date").isNotNull(),datediff(col("order_delivered_customer_date"),col("order_estimated_delivery_date"))).otherwise(None))
           
        elif table_name == 'orderitems': 
            df = df.withColumn("total_item_value",round(col("freight_value")+col("price"),3)) 

        df.write.format('delta').mode('overwrite').option('overwriteSchema',True).save(f"{silver_path}/{table_name}")     
        spark.sql(f"create table if not exists {database}.{table_name} using delta location '{silver_path}/{table_name}'")
        return df
  
    except Exception as e:
        print(f"failed to read {table_name}")
        print(f"failed with {str(e)}")
        return None

 
df_silver_customer = reading_bronze_files(table_name='customer')
df_silver_geolocation = reading_bronze_files(table_name='geolocation')
df_silver_orderitems = reading_bronze_files(table_name='orderitems')
df_silver_orderpayments = reading_bronze_files(table_name='orderpayments')
df_silver_orderreviews = reading_bronze_files(table_name='orderreviews')
df_silver_orders = reading_bronze_files(table_name='orders')
df_silver_products = reading_bronze_files(table_name='products')
df_silver_sellers = reading_bronze_files(table_name='sellers')
df_silver_productcategory = reading_bronze_files(table_name='product_category')


def check_nulls(df, table_name):
    df.select([
        sum(when(col(c).cast("string").isNull() |(col(c).cast("string") == ""), 1) .otherwise(0)).alias(c)
        for c in df.columns
    ]).show(vertical=True)

check_nulls(df_silver_customer, "customer")
check_nulls(df_silver_geolocation,  "geolocation")
check_nulls(df_silver_orderitems, "orderitems") 
check_nulls(df_silver_orderpayments, "orderpayments")
check_nulls(df_silver_orderreviews,  "orderreviews")
check_nulls(df_silver_orders, "orders") 
check_nulls(df_silver_products, "products")
check_nulls(df_silver_sellers,  "sellers")
check_nulls(df_silver_productcategory, "product_category") 


**checking order status**

In [0]:
df_orders.groupBy('order_status').agg(count('*').alias('count')).orderBy('count',ascending =False).show()

In [0]:
df_orders = df_orders.withColumn("is_delivered",when(col('order_status') == 'delivered',True).otherwise(False))\
         .withColumn("delivery_delay_days",when(col("order_delivered_customer_date").isNotNull(),datediff(col("order_delivered_customer_date"),col("order_estimated_delivery_date"))).otherwise(None))


df_orders.write.format('delta').mode('overwrite').option('overwriteSchema',True).save(f"{silver_path}/orders")     

spark.sql(f"create table if not exists {database}.orders using delta location '{silver_path}/orders'")

df_items = df_items.withColumn("total_item_value",round(col("freight_value")+col("price"),3)) 

df_items.write.format('delta').mode('overwrite').option('overwriteSchema',True).save(f"{silver_path}/orderitems") 

spark.sql(f"create table if not exists {database}.orders using delta location '{silver_path}/orderitems'")



